In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# ============ LOAD ============
df = pd.read_excel(r"C:\Users\Jatin Khandelwal\Downloads\Ground water\data\processed\rajasthan_groundwater_merged.xlsx")  # apna path daalo
print("Original shape:", df.shape)

# ============ 1. MISSING VALUE % CHECK ============
missing_pct = (df.isna().sum() / len(df)) * 100
print("\nMissing % per column:\n", missing_pct)

# Drop columns with more than 40% missing
threshold = 40
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
print("\nDropping (too many missing):", cols_to_drop)
df = df.drop(columns=cols_to_drop)

# ============ 2. ZERO/LOW VARIANCE CHECK (categorical) ============
for col in df.select_dtypes(include="object").columns:
    print(f"{col}: {df[col].nunique()} unique values")

# Drop columns with only 1 unique value (no variance = useless for prediction)
zero_var_cols = [col for col in df.select_dtypes(include="object").columns if df[col].nunique() == 1]
print("\nDropping (zero variance):", zero_var_cols)
df = df.drop(columns=zero_var_cols)

# ============ 3. CLEAN REMAINING DATA ============
df["DEPTH"] = pd.to_numeric(df["DEPTH"], errors="coerce")  # force numeric, bad values -> NaN
df["WLS_DATE"] = pd.to_datetime(df["WLS_DATE"], format="%d-%b-%y")
df["month"] = df["WLS_DATE"].dt.month
df["year"] = df["WLS_DATE"].dt.year

df = df.dropna(subset=["DEPTH", "WLS_WTR_LEVEL"])  # drop rows missing key values
print("\nShape after cleaning:", df.shape)

# ============ 4. CORRELATION CHECK (numeric features vs target) ============
numeric_cols = ["DEPTH", "month", "year", "WLS_WTR_LEVEL"]
print("\nCorrelation with target:\n", df[numeric_cols].corr()["WLS_WTR_LEVEL"])

# ============ 5. ENCODING (categorical -> numeric) ============
categorical_cols = df.select_dtypes(include="object").columns.tolist()
# Remove ID/name columns that shouldn't be encoded as features
categorical_cols = [c for c in categorical_cols if c not in ["SITE_ID", "SITE_NAME", "TAHSIL_NAME", "BLOCK_NAME"]]
print("\nEncoding these categorical columns:", categorical_cols)

df_encoded = pd.get_dummies(df, columns=categorical_cols)

# ============ 6. BUILD FEATURES (X) AND TARGET (y) ============
drop_for_X = ["WLS_WTR_LEVEL", "SITE_ID", "SITE_NAME", "TAHSIL_NAME", "BLOCK_NAME", "WLS_DATE"]
drop_for_X = [c for c in drop_for_X if c in df_encoded.columns]
X = df_encoded.drop(columns=drop_for_X)
y = df_encoded["WLS_WTR_LEVEL"]

# ============ 7. TRAIN/TEST SPLIT + MODEL ============
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ============ 8. EVALUATE ============
y_pred = model.predict(X_test)
print("\nMAE:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

# ============ 9. FEATURE IMPORTANCE (for demo) ============
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 10 important features:\n", importance.head(10))

# ============ 10. SAMPLE PREDICTION (for demo) ============
sample = X_test.iloc[[0]]
print(f"\nSample -> Predicted: {model.predict(sample)[0]:.2f} m | Actual: {y_test.iloc[0]:.2f} m")

Original shape: (2605, 13)

Missing % per column:
 FREQUENCY TABLE OF DEPTH TO WATER LEVEL (m bgl) , MAY 2019-RAJASTHAN    100.000000
SITE_ID                                                                   0.000000
STATE_NAME                                                                0.000000
DISTRICT_NAME                                                             0.000000
TAHSIL_NAME                                                               0.000000
BLOCK_NAME                                                                0.000000
SITE_NAME                                                                 0.000000
SITE_TYPE                                                                 0.000000
SITE_SUB_TYPE                                                            67.523992
AQUIFER_TYPE                                                              0.000000
DEPTH                                                                     0.000000
WLS_DATE                            

In [21]:
# 5 alag samples dikhao, sirf 1 nahi
print("=== Sample Predictions (Test Data) ===\n")
for i in range(5):
    sample = X_test.iloc[[i]]
    predicted = model.predict(sample)[0]
    actual = y_test.iloc[i]
    error = abs(predicted - actual)
    print(f"Sample {i+1}: Predicted = {predicted:.2f}m | Actual = {actual:.2f}m | Error = {error:.2f}m")

=== Sample Predictions (Test Data) ===

Sample 1: Predicted = 2.04m | Actual = 4.15m | Error = 2.11m
Sample 2: Predicted = 3.97m | Actual = 2.61m | Error = 1.36m
Sample 3: Predicted = 1.24m | Actual = 0.20m | Error = 1.04m
Sample 4: Predicted = 7.33m | Actual = 10.05m | Error = 2.72m
Sample 5: Predicted = 8.01m | Actual = 10.36m | Error = 2.35m


In [ ]:
def predict_water_level(district, site_type, depth, month, year):
    
    input_data = pd.DataFrame(0, index=[0], columns=X.columns)
    
  
    input_data["DEPTH"] = depth
    input_data["month"] = month
    input_data["year"] = year
    

    district_col = f"DISTRICT_NAME_{district.upper()}"
    sitetype_col = f"SITE_TYPE_{site_type}"
    
    if district_col in input_data.columns:
        input_data[district_col] = 1
    else:
        print(f"Warning: {district} district data mein nahi mila, default use ho raha hai")
    
    if sitetype_col in input_data.columns:
        input_data[sitetype_col] = 1
    else:
        print(f"Warning: {site_type} site type data mein nahi mila")
    
    prediction = model.predict(input_data)[0]
    return prediction


result = predict_water_level(district="AJMER", site_type="Dug Well", depth=20, month=5, year=2023)
print(f"\nPredicted water level for Ajmer (Dug Well, 20m depth, May): {result:.2f} meters")


result2 = predict_water_level(district="JAIPUR", site_type="Bore Well", depth=50, month=8, year=2023)
print(f"Predicted water level for Jaipur (Bore Well, 50m depth, August): {result2:.2f} meters")


Predicted water level for Ajmer (Dug Well, 20m depth, May): 13.13 meters
Predicted water level for Jaipur (Bore Well, 50m depth, August): 10.97 meters


In [ ]:
def get_advisory_depth(district, block=None):
    """
    Historical data se borewell depth advisory suggest karta hai.
    Pehle block-level try karta hai, phir district-level pe fallback.
    """
    
    district = district.upper().strip()   # case/spacing ki galti se bachne ke liye
    
   
    if district not in df['DISTRICT_NAME'].str.upper().unique():
        return None, f"'{district}' naam ka district dataset mein nahi mila. Spelling check karo."
    
    district_data = df[df['DISTRICT_NAME'].str.upper() == district]
    
    if block:
        block = block.upper().strip()
        block_data = district_data[district_data['BLOCK_NAME'].str.upper() == block]
        
        if len(block_data) >= 3:
            avg_depth = block_data['DEPTH'].mean()
            return round(avg_depth, 2), f"Block-level estimate ({len(block_data)} historical wells mile, reliable)"
        
        elif len(block_data) >= 1:
            avg_depth = block_data['DEPTH'].mean()
            return round(avg_depth, 2), f"Block-level estimate (sirf {len(block_data)} historical well{'s' if len(block_data)>1 else ''} mila, kam data — caution ke saath use karo)"
    
    if len(district_data) == 0:
        return None, "Is district ke liye koi bhi historical data nahi hai."
    
    avg_depth = district_data['DEPTH'].mean()
    return round(avg_depth, 2), f"District-level estimate ({len(district_data)} historical wells, block match nahi mila ya diya nahi gaya)"

In [24]:
# Test 1: High-data block (jaise PHAGI, jisme 90 rows the)
depth, note = get_advisory_depth("JAIPUR", "PHAGI")
print(f"Test 1 — Jaipur/Phagi: {depth}m — {note}")

# Test 2: Low-data block (jaise wo blocks jisme sirf 1 row thi)
depth, note = get_advisory_depth("BUNDI", "BUNDI")
print(f"Test 2 — Bundi/Bundi: {depth}m — {note}")

# Test 3: Block hi nahi diya — sirf district
depth, note = get_advisory_depth("AJMER")
print(f"Test 3 — Sirf Ajmer district: {depth}m — {note}")

# Test 4: Galat district naam (error handling check)
depth, note = get_advisory_depth("XYZDISTRICT")
print(f"Test 4 — Invalid district: {depth} — {note}")

Test 1 — Jaipur/Phagi: 48.93m — Block-level estimate (90 historical wells mile, reliable)
Test 2 — Bundi/Bundi: 7.5m — Block-level estimate (sirf 1 historical well mila, kam data — caution ke saath use karo)
Test 3 — Sirf Ajmer district: 38.92m — District-level estimate (84 historical wells, block match nahi mila ya diya nahi gaya)
Test 4 — Invalid district: None — 'XYZDISTRICT' naam ka district dataset mein nahi mila. Spelling check karo.


In [ ]:
df[['DISTRICT_NAME', 'BLOCK_NAME', 'DEPTH']].to_csv(
    r"C:\Users\Jatin Khandelwal\Downloads\Ground water\models\borewell_lookup.csv",
    index=False
)